In [1]:
import pandas as pd
import numpy as np
import re
import warnings 
warnings.filterwarnings('ignore')
from sklearn.base import BaseEstimator, TransformerMixin

In [2]:
df_raw = pd.read_csv('base_dados_passos_magicos.csv', sep=';')

In [3]:
df_raw.head()

,RA,Ano_avaliacao,Nome_aluno,Idade,Fase,Turma,Instituicao_ensino,INDE,Pedra_atual,Pedra 20,...,Delta_IAN,Evolucao_INDE,Evolucao_IDA,Evolucao_IEG,Evolucao_IAA,Evolucao_IPS,Evolucao_IPP,Evolucao_IPV,Evolucao_IAN,Persona_aluno
0,RA-1,2022,Aluno-1,19,Fase 7,A,Escola Pública,5.783,Quartzo,Ametista,...,NaN,Primeiro ano,Primeiro ano,Primeiro ano,Primeiro ano,Primeiro ano,Primeiro ano,Primeiro ano,Primeiro ano,Alerta acadêmico (desfocados)
1,RA-1,2023,Aluno-1,20,Fase 8,8E,Privada - Parcerias com Bolsa 100%,NaN,Sem registro,Ametista,...,5.0,Sem dados,Sem dados,Sem dados,Sem dados,Sem dados,Sem dados,Sem dados,Avanço,Dados incompletos para perfil
2,RA-1,2024,Aluno-1,21,Fase 8,8E,Privada - Parcerias com Bolsa 100%,NaN,Sem registro,Ametista,...,0.0,Sem dados,Sem dados,Sem dados,Sem dados,Sem dados,Sem dados,Sem dados,Neutra,Dados incompletos para perfil
3,RA-10,2022,Aluno-10,18,Fase 7,A,Escola Pública,5.784,Quartzo,Pré-ingresso,...,NaN,Primeiro ano,Primeiro ano,Primeiro ano,Primeiro ano,Primeiro ano,Primeiro ano,Primeiro ano,Primeiro ano,Alerta acadêmico (desfocados)
4,RA-100,2022,Aluno-100,13,Fase 4,A,Rede Decisão,7.618,Ametista,Ametista,...,NaN,Primeiro ano,Primeiro ano,Primeiro ano,Primeiro ano,Primeiro ano,Primeiro ano,Primeiro ano,Primeiro ano,Esforçados em risco emocional


## TARGET E REMOÇÃO DE LEAKAGE

### TARGET
IAN assume 3 valores: 2.5 (defasagem grave), 5.0 (moderada), 10.0 (ok)

* em_risco = 1 → aluno com IAN <= 5 (abaixo da fase esperada) OU IDA < 5.0 (desempenho acadêmico crítico)
* em_risco = 0 → aluno na fase correta E com desempenho adequado

In [4]:
df = df_raw.copy()
df['em_risco'] = ((df['IAN'] <= 5.0) |
                  (df['IDA'] < 5.0)) .astype(int)

print("── Distribuição do target ──")
print(df['em_risco'].value_counts())
print(f"Balanceamento: {df['em_risco'].mean():.1%} em risco\n")
print("Taxa de risco por ano (tendência do programa):")
print(df.groupby('Ano_avaliacao')['em_risco'].mean().map('{:.1%}'.format))

── Distribuição do target ──
em_risco
1    1893
0    1137
Name: count, dtype: int64
Balanceamento: 62.5% em risco

Taxa de risco por ano (tendência do programa):
Ano_avaliacao
2022    75.0%
2023    58.3%
2024    56.8%
Name: em_risco, dtype: object


## REMOÇÃO DE LEAKAGE 
* Qualquer coluna derivada do IAN ou que o contenha precisa sair.
* Se deixar, o modelo "trapaceia" — aprende a virar o gabarito, não aprende os padrões reais dos indicadores.

In [5]:
colunas_remover = [
    # leakage direto do target
    'IAN', 'Defasagem', 'Fase ideal', 'INDE',
    'Delta_IAN', 'Evolucao_IAN',
    # sem sinal preditivo para input novo
    'RA', 'Nome_aluno', 'Turma',
    'Atingiu PV', 'Atingiu_PV_calculado',
    'Pedra 20', 'Pedra 21',
    'Nota_ing',        # 64% nulos, pouco sinal adicional ao IDA
    # versões texto das Deltas (já temos o número)
    'Evolucao_INDE', 'Evolucao_IDA', 'Evolucao_IEG',
    'Evolucao_IAA', 'Evolucao_IPS', 'Evolucao_IPP', 'Evolucao_IPV',
]

colunas_existentes = [c for c in colunas_remover if c in df.columns]
df = df.drop(columns=colunas_existentes)

print(f"\nColunas removidas: {len(colunas_existentes)}")
print(f"Colunas restantes: {df.shape[1]}")


Colunas removidas: 21
Colunas restantes: 22


## CLASSE: LimpadorBase

* O que faz: extrai Fase_num a partir do texto da coluna Fase.

* Quando o Streamlit receber "Fase 3", precisa converter para 3 da mesma forma que foi feito no treino.

* fit()       → não precisa aprender nada (é uma regra fixa)
* transform() → aplica a conversão em qualquer DataFrame

In [6]:
class LimpadorBase(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.fases_conhecidas_ = set(X['Fase'].dropna().unique())
        return self

    def transform(self, X):
        
        X =X.copy()
        
        def _fase_para_numero(valor):
            if pd.isna(valor):
                return np.nan
            v =str(valor).strip().upper()
            if 'ALFA' in v:
                return 0
            match = re.search(r'\d', v)
            if match:
                return int(match.group())
            # Chegou aqui = valor sem 'ALFA' e sem dígito
            # Ex: '', 'Sem fase', 'None' → NaN em vez de quebrar
            return np.nan

        X['Fase_num'] = X['Fase'].apply(_fase_para_numero)

        # Avisa sobre valores não vistos no treino (não quebra, mas registra)

        desconhecidos = set(X['Fase'].dropna().unique()) - self.fases_conhecidas_
        if desconhecidos:
            print(f"Fases não vistas no treino: {desconhecidos} → virarão NaN")
        else:
            print(f"Fases válidas: {sorted(self.fases_conhecidas_)}")

        return X                                    

### Teste isolado da classe

In [7]:
limpador = LimpadorBase()
df = limpador.fit_transform(df)
 
print("── LimpadorBase ──")
print(df.groupby('Fase')['Fase_num'].first().sort_values().to_string())
print(f"Nulos em Fase_num: {df['Fase_num'].isna().sum()}")

Fases válidas: ['Alfa', 'Fase 1', 'Fase 2', 'Fase 3', 'Fase 4', 'Fase 5', 'Fase 6', 'Fase 7', 'Fase 8', 'Fase 9']
── LimpadorBase ──
Fase
Alfa      0
Fase 1    1
Fase 2    2
Fase 3    3
Fase 4    4
Fase 5    5
Fase 6    6
Fase 7    7
Fase 8    8
Fase 9    9
Nulos em Fase_num: 0


In [8]:
df_teste_fase = pd.DataFrame({'Fase': ['Alfa', 'Fase 3', 'Fase Especial', None]})
print("\nTeste com valor desconhecido:")
resultado = limpador.transform(df_teste_fase)
print(resultado[['Fase', 'Fase_num']].to_string(index=False))


Teste com valor desconhecido:
Fases não vistas no treino: {'Fase Especial'} → virarão NaN
         Fase  Fase_num
         Alfa       0.0
       Fase 3       3.0
Fase Especial       NaN
         None       NaN


## CLASSE: ImputadorIndicadores

* O que faz: imputa nulos em IDA, IEG, IAA, IPS, IPV usando a mediana do grupo Fase + Ano_avaliacao

* Por que fit() e transform() são separados aqui?
  * O fit() APRENDE as medianas olhando APENAS o treino.
  * O transform() APLICA essas medianas aprendidas no teste ou input novo.

In [9]:
class ImputadorIndicadores(BaseEstimator, TransformerMixin):
    def __init__(self,indicadores=None):
        self.indicadores = indicadores or ['IDA', 'IEG', 'IAA', 'IPS', 'IPV']

    def fit(self, X, y=None):
        self.medianas_grupo_ = {}
        self.medianas_global_ = {}

        for col in self.indicadores:
            if col not in X.columns:
                continue
            #Mediana por fase + ano
            self.medianas_grupo_[col] = (X.groupby(['Fase', 'Ano_avaliacao'])[col].median().to_dict())
            #Fallback: mediana global (para grupos não vistos)
            self.medianas_global_[col] = X[col].median()

        return self

    def transform(self, X):
        X = X.copy()

        for col in self.indicadores:
            if col not in X.columns:
                continue
            if X[col].isna().sum() == 0:
                continue

            def _imputa(row):
                if pd.notna(row[col]):
                    return row[col]
                chave = (row['Fase'], row['Ano_avaliacao'])
                return self.medianas_grupo_[col].get(chave, self.medianas_global_[col])

            X[col] = X.apply(_imputa, axis = 1)
             #Fallback final (grupo não visto no treino)
            X[col] = X[col].fillna(self.medianas_global_[col])

        return X

### Teste isolado da classe

In [10]:
imputer_ind = ImputadorIndicadores()
df = imputer_ind.fit_transform(df)
 
print("── ImputadorIndicadores ──")
for col in ['IDA', 'IEG', 'IAA', 'IPS', 'IPV']:
    print(f"  {col}: {df[col].isna().sum()} nulos restantes")

── ImputadorIndicadores ──
  IDA: 0 nulos restantes
  IEG: 0 nulos restantes
  IAA: 0 nulos restantes
  IPS: 0 nulos restantes
  IPV: 0 nulos restantes


## CLASSE: ImputadorIPP

* O que faz: imputa IPP com mediana por Fase (não por Fase+Ano, porque IPP tem 34% de nulos — grupos ficariam muito pequenos com 3 dimensões). Cria flag ipp_disponivel ANTES de imputar.
* Fases 8 e 9 têm 100% de nulos no IPP → usam mediana global.

In [11]:
class ImputadorIPP(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # Aprende mediana por Fase (apenas onde há dado real)
        self.mediana_por_fase_ = (
            X.groupby('Fase')['IPP']
            .median()
            .to_dict()
        )
        # Fallback global
        self.mediana_global_ = X['IPP'].median()

        print("Medianas de IPP aprendidas por Fase:")
        for fase, med in sorted(self.mediana_por_fase_.items()):
            status = f"{med:.3f}" if pd.notna(med) else "NaN → usa global"
            print(f"  {fase:<10} {status}")
        print(f"  Global: {self.mediana_global_:.3f}")
        return self

    def transform(self, X):
        X = X.copy()
 
        # Flag ANTES de imputar
        X['ipp_disponivel'] = X['IPP'].notna().astype(int)
 
        def _imputa_ipp(row):
            if pd.notna(row['IPP']):
                return row['IPP']
            mediana_fase = self.mediana_por_fase_.get(row['Fase'], np.nan)
            if pd.notna(mediana_fase):
                return mediana_fase
            return self.mediana_global_
 
        X['IPP'] = X.apply(_imputa_ipp, axis=1)
        return X

### Teste isolado da classe 

In [12]:
imputer_ipp = ImputadorIPP()
df = imputer_ipp.fit_transform(df)
 
print("\n── ImputadorIPP ──")
print(f"IPP nulos restantes: {df['IPP'].isna().sum()}")
print(f"ipp_disponivel — com dado original: {df['ipp_disponivel'].sum()} ({df['ipp_disponivel'].mean():.1%})")

Medianas de IPP aprendidas por Fase:
  Alfa       7.500
  Fase 1     7.917
  Fase 2     7.917
  Fase 3     7.500
  Fase 4     7.812
  Fase 5     7.500
  Fase 6     7.812
  Fase 7     7.500
  Fase 8     NaN → usa global
  Fase 9     NaN → usa global
  Global: 7.500

── ImputadorIPP ──
IPP nulos restantes: 0
ipp_disponivel — com dado original: 1992 (65.7%)


## CLASSE: EncoderCategorico

* O que faz: converte Pedra_atual e Persona_aluno para número.

* Pedra → encoding ORDINAL (tem hierarquia confirmada pelos dados)
   Quartzo=1 < Ágata=2 < Ametista=3 < Topázio=4
   INDE médio: 5.37 → 6.59 → 7.52 → 8.44 ✓

* Persona → encoding NOMINAL (sem hierarquia — são clusters); Tratou-se como número mas o modelo tree não assume ordem

* Por que o fit() importa aqui?
     * Garante que o mapeamento aprendido no treino é o mesmo aplicado em produção. Se uma categoria nova aparecer (ex: nova pedra), o transform() trata como NaNem vez de quebrar silenciosamente.

In [13]:
class EncoderCategorico(BaseEstimator, TransformerMixin):
        mapa_pedra = {
        'Quartzo'     : 1,
        'Ágata'       : 2,
        'Ametista'    : 3,
        'Topázio'     : 4,
        'Sem registro': np.nan,  # tratado separadamente abaixo
    }
 
        mapa_persona = {
        'Alto desempenho (estrelas resilientes)' : 0,
        'Alerta acadêmico (desfocados)'          : 1,
        'Esforçados em risco emocional'          : 2,
        'Crise de autopercepção (invisíveis)'    : 3,
        'Dados incompletos para perfil'          : np.nan,
    }

        def fit(self, X, y=None):
            # Registra as categorias vistas no treino para cada coluna.
            # Se a ONG adicionar uma nova pedra ou persona no futuro,
            # o transform() vai avisar em vez de falhar silenciosamente.
            self.pedras_conhecidas_ = set(X['Pedra_atual'].dropna().unique())
            self.personas_conhecidas_ = set(X['Persona_aluno'].dropna().unique())
        
            pedra_temp = X['Pedra_atual'].map(self.mapa_pedra)
            self.mediana_pedra_ = pedra_temp.median()
 
            return self

        def transform(self, X):
            X = X.copy()
            desconhecidas_pedra = (set(X['Pedra_atual'].dropna().unique()) - self.pedras_conhecidas_)
            if desconhecidas_pedra:
                print(f"Pedras não vistas no treino: {desconhecidas_pedra} → NaN")

            X['Pedra_enc'] = (X['Pedra_atual'].map(self.mapa_pedra).fillna(self.mediana_pedra_)   # 'Sem registro' → mediana do treino
                                )
            desconhecidas_persona = (set(X['Persona_aluno'].dropna().unique()) - self.personas_conhecidas_)

            if desconhecidas_persona:
                print(f"Personas não vistas no treino: {desconhecidas_persona} → NaN")
 
            X['Persona_enc'] = X['Persona_aluno'].map(self.mapa_persona)
            # NaN para perfil incompleto — o modelo tree lida nativamente
 
            return X

###  Teste isolado da classe

In [14]:
encoder = EncoderCategorico()
df = encoder.fit_transform(df)
 
print("── EncoderCategorico ──")
print("Pedra_enc:")
print(df.groupby('Pedra_atual')['Pedra_enc'].first().sort_values().to_string())
print(f"\nPersona_enc nulos (perfil incompleto): {df['Persona_enc'].isna().sum()}")

── EncoderCategorico ──
Pedra_enc:
Pedra_atual
Quartzo         1.0
Ágata           2.0
Ametista        3.0
Sem registro    3.0
Topázio         4.0

Persona_enc nulos (perfil incompleto): 185


In [15]:
df_teste_enc = pd.DataFrame({
    'Pedra_atual'  : ['Quartzo', 'Topázio', 'Diamante'],
    'Persona_aluno': ['Alerta acadêmico (desfocados)',
                      'Dados incompletos para perfil',
                      'Novo perfil experimental'],
})
print("\nTeste com categorias desconhecidas:")
resultado = encoder.transform(df_teste_enc)
print(resultado[['Pedra_atual', 'Pedra_enc', 'Persona_aluno', 'Persona_enc']].to_string(index=False))


Teste com categorias desconhecidas:
Pedras não vistas no treino: {'Diamante'} → NaN
Personas não vistas no treino: {'Novo perfil experimental'} → NaN
Pedra_atual  Pedra_enc                 Persona_aluno  Persona_enc
    Quartzo        1.0 Alerta acadêmico (desfocados)          1.0
    Topázio        4.0 Dados incompletos para perfil          NaN
   Diamante        3.0      Novo perfil experimental          NaN


## CLASSE: CriadorFeatures

* O que faz: cria 3 features derivadas que capturam relações entre indicadores.

* gap_percepcao_realidade: IAA - IDA
    * IAA alto + IDA baixo = aluno acha que vai bem mas não vai → "efeito Dunning-Kruger" → risco não percebido

* indice_bem_estar: (IPS + IAA) / 2
    * Combina o lado externo (psicossocial) com o interno (auto) → sinal composto de saúde emocional

* pressao_psicossocial: IEG - IPS
    * Aluno muito engajado mas com IPS baixo = esforço sem suporte → risco de burnout / colapso futuro

In [16]:
class CriadorFeatures(BaseEstimator, TransformerMixin): 
    def fit(self, X, y=None):
        return self   # regras fixas, nada para aprender
 
    def transform(self, X):
        X = X.copy()
 
        # Aluno acha que vai bem, mas vai mal
        X['gap_percepcao_realidade'] = X['IAA'] - X['IDA']
 
        # Saúde emocional composta
        X['indice_bem_estar'] = (X['IPS'] + X['IAA']) / 2
 
        # Alto esforço sem suporte psicossocial
        X['pressao_psicossocial'] = X['IEG'] - X['IPS']
 
        return X

### Teste isolado da classe

In [17]:
criador = CriadorFeatures()
df = criador.fit_transform(df)
 
print("── CriadorFeatures ──")
for feat, desc in [
    ('gap_percepcao_realidade', 'IAA - IDA (falsa confiança)'),
    ('indice_bem_estar',        '(IPS + IAA) / 2'),
    ('pressao_psicossocial',    'IEG - IPS (esforço sem suporte)'),
]:
    print(f"  {feat:<30} média={df[feat].mean():.2f}  std={df[feat].std():.2f}  |  {desc}")

── CriadorFeatures ──
  gap_percepcao_realidade        média=1.54  std=3.01  |  IAA - IDA (falsa confiança)
  indice_bem_estar               média=7.15  std=1.67  |  (IPS + IAA) / 2
  pressao_psicossocial           média=1.62  std=2.93  |  IEG - IPS (esforço sem suporte)


## CLASSE: SeletorColunas

* O que faz: garante que o DataFrame final tem EXATAMENTE as colunas que o modelo espera, na ordem certa.

* Por que isso importa?
    * O modelo treinado com [IDA, IEG, IPV, ...] vai quebrar se receber [IPV, IDA, IEG, ...] — mesmo dado, ordem errada. Ou quebrar silenciosamente se uma coluna estiver faltando.

* O SeletorColunas é o "contrato" entre o pipeline e o modelo: garante que a entrada do modelo é sempre idêntica ao treino.

* fit()       → registra a lista oficial de features
* transform() → filtra e reordena qualquer DataFrame recebido

In [18]:
class SeletorColunas(BaseEstimator, TransformerMixin):
    features = [
        # Indicadores núcleo
        'IDA', 'IEG', 'IAA', 'IPS', 'IPV', 'IPP',
        # Notas brutas
        'Nota_mat', 'Nota_port',
        # Contexto
        'Idade', 'Fase_num', 'Pedra_enc',
        # Flag de disponibilidade
        'ipp_disponivel',
        # Dinâmica temporal (NaN onde não há histórico — ok para trees)
        'Delta_IDA', 'Delta_IEG', 'Delta_IPS', 'Delta_IAA', 'Delta_IPV',
        'tem_historico_IDA',
        # Features derivadas
        'gap_percepcao_realidade', 'indice_bem_estar', 'pressao_psicossocial',
        # Cluster
        'Persona_enc',
    ]
 
    def fit(self, X, y=None):
        # Verifica quais features da lista estão disponíveis no DataFrame
        self.features_disponiveis_ = [f for f in self.features if f in X.columns]
        ausentes = [f for f in self.features if f not in X.columns]
        if ausentes:
            print(f"Features ausentes (serão ignoradas): {ausentes}")
        print(f"Features selecionadas: {len(self.features_disponiveis_)}")
        return self
 
    def transform(self, X):
        return X[self.features_disponiveis_].copy()
 
    def get_feature_names_out(self):
        """Necessário para compatibilidade com SHAP e outros."""
        return self.features_disponiveis_
 
 
# ── Cria flag de histórico antes do seletor ──────────────────
# (Delta_* já existem na base limpa, mas precisa-se do flag)
delta_cols = ['Delta_IDA', 'Delta_IEG', 'Delta_IPS', 'Delta_IAA', 'Delta_IPV']
for col in delta_cols:
    if col in df.columns:
        flag = col.replace('Delta_', 'tem_historico_')
        df[flag] = df[col].notna().astype(int)
 
# ── Imputa Nota_mat e Nota_port (6% nulos) ───────────────────
for col in ['Nota_mat', 'Nota_port']:
    if col in df.columns:
        df[col] = df.groupby(['Fase', 'Ano_avaliacao'])[col].transform(
            lambda x: x.fillna(x.median())
        )
        df[col] = df[col].fillna(df[col].median())

### Teste isolado do seletor

In [19]:
seletor = SeletorColunas()
df_features = seletor.fit_transform(df)
 
print("── SeletorColunas ──")
print(f"Shape final das features: {df_features.shape}")
print(f"\nNulos por feature:")
nulos = df_features.isna().sum()
nulos_com = nulos[nulos > 0]
if len(nulos_com) > 0:
    for col, n in nulos_com.items():
        print(f"  {col:<30} {n} ({n/len(df_features):.1%})")
    print("  (NaN nas Deltas é esperado — alunos sem histórico)")
else:
    print("  Nenhum nulo nas features fixas ✓")

Features selecionadas: 22
── SeletorColunas ──
Shape final das features: (3030, 22)

Nulos por feature:
  Delta_IDA                      1770 (58.4%)
  Delta_IEG                      1759 (58.1%)
  Delta_IPS                      1760 (58.1%)
  Delta_IAA                      1753 (57.9%)
  Delta_IPV                      1771 (58.4%)
  Persona_enc                    185 (6.1%)
  (NaN nas Deltas é esperado — alunos sem histórico)


## VALIDAÇÃO FINAL E SPLIT

In [20]:
target = 'em_risco'
 
print("── Verificação de leakage ──")
correlacoes = df_features.corrwith(df[target]).abs().sort_values(ascending=False)
print("Top 10 correlações com o target:")
print(correlacoes.head(10).map('{:.3f}'.format).to_string())
if correlacoes.max() > 0.95:
    print("Possível leakage!")
else:
    print("\n✅ Nenhuma correlação suspeita (> 0.95)")

── Verificação de leakage ──
Top 10 correlações com o target:
Pedra_enc                  0.495
IDA                        0.340
Nota_mat                   0.271
Nota_port                  0.260
Fase_num                   0.244
IPV                        0.244
gap_percepcao_realidade    0.154
IPP                        0.125
Persona_enc                0.098
Delta_IPS                  0.090

✅ Nenhuma correlação suspeita (> 0.95)


In [21]:
anos = df['Ano_avaliacao']
 
X = df_features.copy()
y = df[target].copy()
 
mask_treino = anos.isin([2022, 2023])
mask_teste  = anos == 2024
 
X_train, y_train = X[mask_treino], y[mask_treino]
X_test,  y_test  = X[mask_teste],  y[mask_teste]
 
print("\n── Split treino / teste ──")
print(f"Treino (2022+2023): {len(X_train):>4} registros | {y_train.mean():.1%} em risco")
print(f"Teste  (2024):      {len(X_test):>4} registros | {y_test.mean():.1%} em risco")
print(f"\n Distribuições diferentes entre treino e teste.")
print(f"   O programa funcionou: risco caiu de {y_train.mean():.1%} para {y_test.mean():.1%}.")
print(f"   Isso é real — o modelo vai prever com base nos padrões")
print(f"   aprendidos, não na taxa de base. Monitorar F1 > accuracy.")
 
print("\n✅ Base pronta. Próximo passo: treinamento dos modelos.")
print(f"\nObjetos disponíveis para o pipeline:")
print(f"  limpador       → LimpadorBase()")
print(f"  imputer_ind    → ImputadorIndicadores()")
print(f"  imputer_ipp    → ImputadorIPP()")
print(f"  encoder        → EncoderCategorico()")
print(f"  criador        → CriadorFeatures()")
print(f"  seletor        → SeletorColunas()")
print(f"  X_train, y_train, X_test, y_test  → prontos para modelagem")


── Split treino / teste ──
Treino (2022+2023): 1874 registros | 66.0% em risco
Teste  (2024):      1156 registros | 56.8% em risco

 Distribuições diferentes entre treino e teste.
   O programa funcionou: risco caiu de 66.0% para 56.8%.
   Isso é real — o modelo vai prever com base nos padrões
   aprendidos, não na taxa de base. Monitorar F1 > accuracy.

✅ Base pronta. Próximo passo: treinamento dos modelos.

Objetos disponíveis para o pipeline:
  limpador       → LimpadorBase()
  imputer_ind    → ImputadorIndicadores()
  imputer_ipp    → ImputadorIPP()
  encoder        → EncoderCategorico()
  criador        → CriadorFeatures()
  seletor        → SeletorColunas()
  X_train, y_train, X_test, y_test  → prontos para modelagem


## MODELOS

In [22]:
#pip install shap

In [23]:
from sklearn.pipeline      import Pipeline
from sklearn.impute        import SimpleImputer
from sklearn.linear_model  import LogisticRegression
from sklearn.ensemble      import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics       import (classification_report, roc_auc_score,
                                   f1_score, recall_score, precision_score,
                                   RocCurveDisplay, ConfusionMatrixDisplay)
from sklearn.model_selection import StratifiedKFold, cross_val_score
import xgboost as xgb
import lightgbm as lgb
import joblib
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os
from sklearn.preprocessing import StandardScaler
import shap

In [24]:
#SPLIT CORRETO: ANTES DAS TRANSFORMAÇÕES


# As classes anteriores foram rodadas no df inteiro para  ver o resultado de cada uma. Mas em produção, o pipeline precisa aprender APENAS com o treino.
# Se fizer fit() no dataset completo, o imputador aprende a mediana do teste também — isso é data leakage sutil.
# A correção: split primeiro, pipeline.fit() só no treino. Por isso voltou-se ao df_raw e recriou o split "limpo".


df_base = df_raw.copy()
df_base['em_risco'] = ((df_base['IAN'] <= 5.0) | (df_base['IDA'] < 5)).astype(int)
df_base = df_base.drop(columns=[c for c in colunas_remover if c in df_base.columns])
 
# Trata Nota_mat e Nota_port (6% nulos — fora do pipeline,
# pois depende de groupby complexo; fazemos antes do split)
for col in ['Nota_mat', 'Nota_port']:
    if col in df_base.columns:
        df_base[col] = df_base.groupby(['Fase', 'Ano_avaliacao'])[col].transform(
            lambda x: x.fillna(x.median())
        )
        df_base[col] = df_base[col].fillna(df_base[col].median())
 
# Flag de histórico (alunos com 2+ anos)
for col in ['Delta_IDA', 'Delta_IEG', 'Delta_IPS', 'Delta_IAA', 'Delta_IPV']:
    if col in df_base.columns:
        df_base[col.replace('Delta_', 'tem_historico_')] = df_base[col].notna().astype(int)
 
# Split temporal — mesmo critério da célula 9
mask_treino = df_base['Ano_avaliacao'].isin([2022, 2023])
mask_teste  = df_base['Ano_avaliacao'] == 2024
 
target = 'em_risco'
 
df_train_raw = df_base[mask_treino].copy()
df_test_raw  = df_base[mask_teste].copy()
 
y_train_raw = df_train_raw.pop(target)
y_test_raw  = df_test_raw.pop(target)
 
print("── Split (dados brutos, pré-pipeline) ──")
print(f"Treino: {len(df_train_raw):>4} registros | {y_train_raw.mean():.1%} em risco")
print(f"Teste:  {len(df_test_raw):>4} registros | {y_test_raw.mean():.1%} em risco")
print(f"\nAgora o pipeline vai aprender APENAS com o treino.")

── Split (dados brutos, pré-pipeline) ──
Treino: 1874 registros | 66.0% em risco
Teste:  1156 registros | 56.8% em risco

Agora o pipeline vai aprender APENAS com o treino.


### Contrução do pipeline

In [25]:
def criar_pipeline(modelo):
    """
    Monta o pipeline completo com todos os transformers + modelo.
    Aceita qualquer classificador sklearn-compatível.
 
    O passo 'imputer_final' usa fill_value=0 para as colunas Delta.
    Isso é semanticamente correto: Delta=0 significa "sem variação",
    que é o melhor valor neutro para alunos sem histórico anterior.
    Esse passo garante que Logistic Regression funcione sem NaN,
    e não prejudica RF/XGBoost que já tratam NaN nativamente.
    """
    return Pipeline([
        ('limpador'      , LimpadorBase()),
        ('imputer_ind'   , ImputadorIndicadores()),
        ('imputer_ipp'   , ImputadorIPP()),
        ('encoder'       , EncoderCategorico()),
        ('criador'       , CriadorFeatures()),
        ('seletor'       , SeletorColunas()),
        ('imputer_final' , SimpleImputer(strategy='constant', fill_value=0)),
        ('scaler'        , StandardScaler()),
        ('modelo'        , modelo),
    ])
 
print("Função criar_pipeline() definida ✓")
print("\nEstrutura do pipeline:")
print("  limpador       → Fase_num")
print("  imputer_ind    → imputa IDA, IEG, IAA, IPS, IPV")
print("  imputer_ipp    → imputa IPP + cria flag ipp_disponivel")
print("  encoder        → Pedra_enc, Persona_enc")
print("  criador        → gap, bem_estar, pressao_psicossocial")
print("  seletor        → 23 features na ordem certa")
print("  scaler         → StandardScaler()")
print("  imputer_final  → Delta NaN → 0 (sem variação)")
print("  modelo         → classificador")

Função criar_pipeline() definida ✓

Estrutura do pipeline:
  limpador       → Fase_num
  imputer_ind    → imputa IDA, IEG, IAA, IPS, IPV
  imputer_ipp    → imputa IPP + cria flag ipp_disponivel
  encoder        → Pedra_enc, Persona_enc
  criador        → gap, bem_estar, pressao_psicossocial
  seletor        → 23 features na ordem certa
  scaler         → StandardScaler()
  imputer_final  → Delta NaN → 0 (sem variação)
  modelo         → classificador


### Treinamento e comparação dos modelos

In [26]:
# Peso para compensar desbalanceamento leve entre classes
_peso = (y_train_raw == 0).sum() / (y_train_raw == 1).sum()
 
modelos_config = {
 

    'Regressão Logística': LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=42,
    ),
 

    'Random Forest': RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=10,
        min_samples_split=20,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1,
    ),
 

    'XGBoost': xgb.XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        min_child_weight=5,
        gamma=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.5,
        reg_lambda=2.0,
        scale_pos_weight=_peso,
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1,
    ),
 
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=20,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        scale_pos_weight=_peso,
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    ),
 

    'HistGradientBoosting': HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.05,
        max_leaf_nodes=20,
        min_samples_leaf=20,
        l2_regularization=1.0,
        class_weight='balanced',
        random_state=42,
    ),
}

os.makedirs('graficos', exist_ok=True)
os.makedirs('modelo', exist_ok=True)

# Treina e avalia cada modelo
resultados = {}
 
print("── Treinando modelos ──\n")
 
for nome, clf in modelos_config.items():
 
    pipe = criar_pipeline(clf)
 
    # Validação cruzada no treino (5 folds estratificados)
    # Dá uma estimativa mais robusta do que um único split
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores_cv = cross_val_score(pipe, df_train_raw, y_train_raw,
                                cv=cv, scoring='recall', n_jobs=-1)
 
    # Treina no treino completo
    pipe.fit(df_train_raw, y_train_raw)
 
    # Métricas no TREINO — para detectar overfitting/underfitting
    y_pred_treino  = pipe.predict(df_train_raw)
    y_proba_treino = pipe.predict_proba(df_train_raw)[:, 1]
 
    # Métricas no TESTE
    y_pred  = pipe.predict(df_test_raw)
    y_proba = pipe.predict_proba(df_test_raw)[:, 1]
 
    resultados[nome] = {
        'pipeline'      : pipe,
        # Treino
        'acuracia_treino': (y_pred_treino == y_train_raw).mean(),
        'recall_treino' : recall_score(y_train_raw, y_pred_treino),
        'f1_treino'     : f1_score(y_train_raw, y_pred_treino),
        'recall_cv'     : scores_cv.mean(),
        'recall_cv_std' : scores_cv.std(),
        # Teste
        'acuracia_teste': (y_pred == y_test_raw).mean(),
        'recall'        : recall_score(y_test_raw, y_pred),
        'precision'     : precision_score(y_test_raw, y_pred),
        'f1'            : f1_score(y_test_raw, y_pred),
        'roc_auc'       : roc_auc_score(y_test_raw, y_proba),
        'y_pred'        : y_pred,
        'y_proba'       : y_proba,
    }
 
    r = resultados[nome]
    print(f"{'─'*45}")
    print(f"  {nome}")
    print(f"  {'':30} {'TREINO':>8}  {'TESTE':>8}  {'DIFF':>8}")
    print(f"  {'Acurácia':<30} {r['acuracia_treino']:>8.3f}  {r['acuracia_teste']:>8.3f}  {r['acuracia_treino']-r['acuracia_teste']:>+8.3f}")
    print(f"  {'Recall':<30} {r['recall_treino']:>8.3f}  {r['recall']:>8.3f}  {r['recall_treino']-r['recall']:>+8.3f}")
    print(f"  {'F1':<30} {r['f1_treino']:>8.3f}  {r['f1']:>8.3f}  {r['f1_treino']-r['f1']:>+8.3f}")
    print(f"  {'Recall CV (validação cruzada)':<30} {r['recall_cv']:>8.3f} ±{r['recall_cv_std']:.3f}")
    print(f"  {'ROC-AUC (teste)':<30} {r['roc_auc']:>8.3f}")
 
    # Diagnóstico automático de overfitting/underfitting
    diff_f1 = r['f1_treino'] - r['f1']
    if diff_f1 > 0.15:
        print(f"⚠️  Overfitting relevante — revisar regularização (+{diff_f1:.3f})")
    elif diff_f1 > 0.08:
        print(f"ℹ️  Diferença dentro do esperado para base pequena "
              f"com shift temporal (+{diff_f1:.3f})")
    else:
        print(f"✅ Generalização excelente") 

── Treinando modelos ──

Fases válidas: ['Alfa', 'Fase 1', 'Fase 2', 'Fase 3', 'Fase 4', 'Fase 5', 'Fase 6', 'Fase 7', 'Fase 8']
Medianas de IPP aprendidas por Fase:
  Alfa       6.875
  Fase 1     7.812
  Fase 2     8.333
  Fase 3     7.656
  Fase 4     7.969
  Fase 5     7.656
  Fase 6     7.812
  Fase 7     7.875
  Fase 8     NaN → usa global
  Global: 7.656
Features selecionadas: 22
Fases válidas: ['Alfa', 'Fase 1', 'Fase 2', 'Fase 3', 'Fase 4', 'Fase 5', 'Fase 6', 'Fase 7', 'Fase 8']
Fases válidas: ['Alfa', 'Fase 1', 'Fase 2', 'Fase 3', 'Fase 4', 'Fase 5', 'Fase 6', 'Fase 7', 'Fase 8']
Fases não vistas no treino: {'Fase 9'} → virarão NaN
Fases não vistas no treino: {'Fase 9'} → virarão NaN
─────────────────────────────────────────────
  Regressão Logística
                                   TREINO     TESTE      DIFF
  Acurácia                          0.906     0.850    +0.056
  Recall                            0.896     0.889    +0.007
  F1                                0.926 

/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fases válidas: ['Alfa', 'Fase 1', 'Fase 2', 'Fase 3', 'Fase 4', 'Fase 5', 'Fase 6', 'Fase 7', 'Fase 8']
Medianas de IPP aprendidas por Fase:
  Alfa       6.875
  Fase 1     7.812
  Fase 2     8.333
  Fase 3     7.656
  Fase 4     7.969
  Fase 5     7.656
  Fase 6     7.812
  Fase 7     7.875
  Fase 8     NaN → usa global
  Global: 7.656
Features selecionadas: 22
Fases válidas: ['Alfa', 'Fase 1', 'Fase 2', 'Fase 3', 'Fase 4', 'Fase 5', 'Fase 6', 'Fase 7', 'Fase 8']
Fases válidas: ['Alfa', 'Fase 1', 'Fase 2', 'Fase 3', 'Fase 4', 'Fase 5', 'Fase 6', 'Fase 7', 'Fase 8']
Fases não vistas no treino: {'Fase 9'} → virarão NaN
Fases não vistas no treino: {'Fase 9'} → virarão NaN
─────────────────────────────────────────────
  LightGBM
                                   TREINO     TESTE      DIFF
  Acurácia                          1.000     0.907    +0.093
  Recall                            1.000     0.951    +0.049
  F1                                1.000     0.920    +0.080
  Recall CV (val

### Tabela comparativa

In [27]:
print("\n── Comparativo final (treino vs teste) ──\n")
print(f"{'Modelo':<25} {'Acur.TR':>8} {'Acur.TE':>8} {'Rec.TR':>8} {'Rec.TE':>8} {'F1.TR':>7} {'F1.TE':>7} {'ROC-AUC':>9}")
print("─" * 88)
 
for nome, r in resultados.items():
    print(f"{nome:<25} "
          f"{r['acuracia_treino']:>8.3f} {r['acuracia_teste']:>8.3f} "
          f"{r['recall_treino']:>8.3f} {r['recall']:>8.3f} "
          f"{r['f1_treino']:>7.3f} {r['f1']:>7.3f} "
          f"{r['roc_auc']:>9.3f}")
 
print("\nLegenda: TR = treino | TE = teste")
print("Overfitting:   métricas TR >> TE  (modelo decorou o treino)")
print("Underfitting:  métricas TR ≈ TE mas ambas baixas (não aprendeu)")
print("Bom:           TR ≈ TE e métricas razoáveis")


── Comparativo final (treino vs teste) ──

Modelo                     Acur.TR  Acur.TE   Rec.TR   Rec.TE   F1.TR   F1.TE   ROC-AUC
────────────────────────────────────────────────────────────────────────────────────────
Regressão Logística          0.906    0.850    0.896    0.889   0.926   0.871     0.923
Random Forest                0.926    0.849    0.917    0.884   0.942   0.869     0.914
XGBoost                      0.993    0.910    0.991    0.944   0.995   0.923     0.944
LightGBM                     1.000    0.907    1.000    0.951   1.000   0.920     0.939
HistGradientBoosting         1.000    0.905    1.000    0.951   1.000   0.919     0.940

Legenda: TR = treino | TE = teste
Overfitting:   métricas TR >> TE  (modelo decorou o treino)
Underfitting:  métricas TR ≈ TE mas ambas baixas (não aprendeu)
Bom:           TR ≈ TE e métricas razoáveis


In [28]:
melhor_modelo = max(resultados, key=lambda n: resultados[n]['f1'])
print(f"\n✅ Modelo selecionado: {melhor_modelo}")
print(f"   Critério: melhor F1 no teste — captura defasagem de fase e desempenho crítico")
print(f"   ROC-AUC: {resultados[melhor_modelo]['roc_auc']:.3f} | F1: {resultados[melhor_modelo]['f1']:.3f}")


✅ Modelo selecionado: XGBoost
   Critério: melhor F1 no teste — captura defasagem de fase e desempenho crítico
   ROC-AUC: 0.944 | F1: 0.923


### Avaliação detalhada do modelo escolhido

In [29]:
melhor = resultados[melhor_modelo]
melhor_pipe = melhor['pipeline']
 
print(f"── Avaliação detalhada: {melhor_modelo} ──\n")
print(classification_report(
    y_test_raw, melhor['y_pred'],
    target_names=['Sem risco (0)', 'Em risco (1)']
))
 
# Matriz de confusão
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
 
ConfusionMatrixDisplay.from_predictions(
    y_test_raw, melhor['y_pred'],
    display_labels=['Sem risco', 'Em risco'],
    cmap='Blues', ax=axes[0]
)
axes[0].set_title(f'Matriz de Confusão — {melhor_modelo}')
 
# Curva ROC
RocCurveDisplay.from_predictions(
    y_test_raw, melhor['y_proba'],
    name=melhor_modelo, ax=axes[1]
)
axes[1].set_title('Curva ROC')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.4)
 
plt.tight_layout()
plt.savefig('graficos/avaliacao_modelo.png', dpi=150, bbox_inches='tight')
plt.show()
print("Gráfico salvo: avaliacao_modelo.png")
 
# Leitura da matriz de confusão em linguagem de negócio
from sklearn.metrics import confusion_matrix
tn, fp, fn, tp = confusion_matrix(y_test_raw, melhor['y_pred']).ravel()
total = len(y_test_raw)
 
print(f"\n── Leitura em linguagem da ONG ──")
print(f"  Alunos em risco identificados corretamente: {tp} "
      f"({tp/(tp+fn):.1%} dos que realmente estão em risco)")
print(f"  Alunos em risco não detectados:             {fn} "
      f"← FALSO NEGATIVO (prioridade minimizar)")
print(f"  Alunos sem risco sinalizados por engano:    {fp} "
      f"← FALSO POSITIVO (menos grave)")
print(f"  Alunos sem risco corretamente liberados:    {tn}")

── Avaliação detalhada: XGBoost ──

               precision    recall  f1-score   support

Sem risco (0)       0.92      0.87      0.89       499
 Em risco (1)       0.90      0.94      0.92       657

     accuracy                           0.91      1156
    macro avg       0.91      0.90      0.91      1156
 weighted avg       0.91      0.91      0.91      1156

Gráfico salvo: avaliacao_modelo.png

── Leitura em linguagem da ONG ──
  Alunos em risco identificados corretamente: 620 (94.4% dos que realmente estão em risco)
  Alunos em risco não detectados:             37 ← FALSO NEGATIVO (prioridade minimizar)
  Alunos sem risco sinalizados por engano:    67 ← FALSO POSITIVO (menos grave)
  Alunos sem risco corretamente liberados:    432


### Coeficientes - padrões que identificam o risco

O que o SHAP diz:
   * Positivo → feature empurrou a predição para risco
   * Negativo → feature empurrou a predição para sem risco
   * Magnitude → quanto maior o valor absoluto, mais impacto naquela predição específica

In [30]:
print("── SHAP: importância das features ──")
 
modelo_final  = melhor_pipe.named_steps['modelo']
feature_names = melhor_pipe.named_steps['seletor'].features_disponiveis_
 
# Dados transformados pelo pipeline (sem o modelo)
X_test_transformado = melhor_pipe[:-1].transform(df_test_raw)
X_test_shap = pd.DataFrame(X_test_transformado, columns=feature_names)

# Calcula SHAP values
explainer   = shap.TreeExplainer(modelo_final)
shap_values = explainer.shap_values(X_test_shap)

# Gráfico de importância global
plt.figure(figsize=(9, 7))
shap.summary_plot(
    shap_values, X_test_shap,
    plot_type='bar',
    show=False,
    max_display=15,
)
plt.title('Importância das Features (SHAP) — impacto médio no risco')
plt.tight_layout()
plt.savefig('graficos/shap_importancia.png', dpi=150, bbox_inches='tight')
plt.show()
print("Gráfico salvo: graficos/shap_importancia.png")

# Importância média por feature
importancia_media = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=feature_names
).sort_values(ascending=False)

print("\n── Top features (SHAP) ──")
for i, (feat, imp) in enumerate(importancia_media.head(10).items(), 1):
    print(f"  {i:2d}. {feat:<30} SHAP médio: {imp:.4f}")

── SHAP: importância das features ──
Fases não vistas no treino: {'Fase 9'} → virarão NaN
Gráfico salvo: graficos/shap_importancia.png

── Top features (SHAP) ──
   1. Fase_num                       SHAP médio: 2.2104
   2. Idade                          SHAP médio: 2.0064
   3. IDA                            SHAP médio: 0.9323
   4. Pedra_enc                      SHAP médio: 0.8235
   5. IPP                            SHAP médio: 0.1258
   6. IEG                            SHAP médio: 0.1214
   7. IPV                            SHAP médio: 0.1212
   8. indice_bem_estar               SHAP médio: 0.1189
   9. Nota_mat                       SHAP médio: 0.0874
  10. IPS                            SHAP médio: 0.0860


### Salva o pipeline e artefatos

In [31]:
nome_modelo = melhor_modelo.lower().replace(' ', '_').replace('ã', 'a').replace('ó', 'o')
 
# Salva o pipeline
joblib.dump(melhor_pipe, 'modelo/pipeline_risco_1.joblib')
print(f"Pipeline salvo: pipeline_risco.joblib")
 
# Salva artefatos de metadados
artefatos = {
    'modelo'         : melhor_modelo,
    'versao'         : '1.0',
    'data_treino'    : '2022-2023',
    'data_teste'     : '2024',
    'features'       : feature_names,
    'n_features'     : len(feature_names),
    'metricas_treino': {
        'acuracia' : round(melhor['acuracia_treino'], 4),
        'recall'   : round(melhor['recall_treino'], 4),
        'f1'       : round(melhor['f1_treino'], 4),
    },
    'metricas_teste' : {
        'acuracia' : round(melhor['acuracia_teste'], 4),
        'recall'   : round(melhor['recall'], 4),
        'precision': round(melhor['precision'], 4),
        'f1'       : round(melhor['f1'], 4),
        'roc_auc'  : round(melhor['roc_auc'], 4),
    },
    'threshold'      : 0.5,
    'target'         : 'em_risco (IAN <= 5)',
    'balanceamento'  : {
        'treino_risco_pct': round(float(y_train_raw.mean()), 4),
        'teste_risco_pct' : round(float(y_test_raw.mean()), 4),
    },
    'top5_features_shap': importancia_media.head(5).index.tolist(),
}
 
with open('modelo/artefatos_modelo_1.json', 'w', encoding='utf-8') as f:
    json.dump(artefatos, f, ensure_ascii=False, indent=2)
 
print(f"Metadados salvos: artefatos_modelo.json")
 
# Confirmação do que foi salvo
print(f"\n── Resumo do que o Streamlit vai usar ──")
print(f"  pipeline_risco.joblib      → carrega com joblib.load()")
print(f"  artefatos_modelo.json   → métricas, threshold, lista de features")
print(f"\n  Para predizer um aluno novo:")
print(f"  pipe = joblib.load('pipeline_risco.joblib')")
print(f"  prob = pipe.predict_proba(df_aluno_novo)[0][1]")
print(f"  → retorna P(em_risco) entre 0 e 1")
 
print(f"\n✅ Modelagem concluída.")
print(f"   Modelo: {melhor_modelo}")
print(f"   Recall: {melhor['recall']:.3f} | F1: {melhor['f1']:.3f} | ROC-AUC: {melhor['roc_auc']:.3f}")

Pipeline salvo: pipeline_risco.joblib
Metadados salvos: artefatos_modelo.json

── Resumo do que o Streamlit vai usar ──
  pipeline_risco.joblib      → carrega com joblib.load()
  artefatos_modelo.json   → métricas, threshold, lista de features

  Para predizer um aluno novo:
  pipe = joblib.load('pipeline_risco.joblib')
  prob = pipe.predict_proba(df_aluno_novo)[0][1]
  → retorna P(em_risco) entre 0 e 1

✅ Modelagem concluída.
   Modelo: XGBoost
   Recall: 0.944 | F1: 0.923 | ROC-AUC: 0.944
